# Melanoma: OME-TIFF -> SpatialData -> precomputed raster + meshes -> Vitessce

Full pipeline for the melanoma dataset, combining conversion and visualization into one notebook. All values below (axis order, camera state, segment IDs/colors) are real, confirmed-working values derived through hands-on debugging, not defaults.

## 1. Setup

In [1]:
%load_ext jupyter_black

In [1]:
from pathlib import Path
from spatialdata import SpatialData
from spatialdata.models import Labels3DModel
from dask_image.imread import imread
import xmltodict
import tifffile

from tissue_map_tools.igneous_converters import (
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes,
)

dataset_path = Path.cwd().parent.parent / "data" / "melanoma"
raw_path = dataset_path / "raw"
out_path = dataset_path / "out"
raw_path.mkdir(parents=True, exist_ok=True)
out_path.mkdir(parents=True, exist_ok=True)
ome_tiff_path = raw_path / "melanoma_mask.ome.tiff"
precomputed_path = out_path / "melanoma_precomputed"

if not ome_tiff_path.exists():
    raise FileNotFoundError(
        f"{ome_tiff_path} does not exist. Please use symlinks to make the data available."
    )

## 2. Load and determine axis order

The heuristic below matches array shape against OME-XML `SizeX/Y/Z` -- confirmed reliable *only* because all three sizes are unique for this file (`SizeX=5454, SizeY=2754, SizeZ=194`). This exact heuristic has been the root cause of a real mesh-rotation bug in the past when sizes collided; the `assert` below guards against silently trusting a wrong result.

In [3]:
data = imread(ome_tiff_path)

xml = tifffile.TiffFile(ome_tiff_path).ome_metadata
xml_dict = xmltodict.parse(xml)
sizes = {
    ax: int(xml_dict["OME"]["Image"]["Pixels"][f"@Size{ax.upper()}"]) for ax in "xyz"
}
assert len(set(sizes.values())) == 3, (
    "Sizes collide -- do not trust the heuristic below."
)

dims = []
for size in data.shape:
    for ax, ax_size in sizes.items():
        if size == ax_size:
            dims.append(ax)
            break
dims = tuple(dims)

print("data.shape =", data.shape)
print("sizes (OME-XML) =", sizes)
print("computed dims =", dims)

data.shape = (194, 2754, 5454)
sizes (OME-XML) = {'x': 5454, 'y': 2754, 'z': 194}
computed dims = ('z', 'y', 'x')


## 3. Build the SpatialData object

No explicit `transformations=` is passed, so physical scale defaults to identity (1 unit = 1 nm downstream). Confirmed real calibration is PhysicalSizeX=PhysicalSizeY=1.0 µm -- pass a real `Scale` transformation here if exact physical scale matters for your use case.

In [ ]:
labels = Labels3DModel.parse(data, dims=dims)
sdata_write_path = out_path / "melanoma_mask.zarr"

sdata_unwritten = SpatialData.init_from_elements({"labels": labels})
sdata_unwritten.write(str(sdata_write_path), overwrite=True)

sdata = SpatialData.read(str(sdata_write_path))

## 4. Convert to precomputed raster + meshes

`parallel=1` is required, not `False` -- `False` is coerced to `0` by Python and crashes the process pool.

In [5]:
if not (precomputed_path / "info").exists():
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes(
        raster=sdata["labels"],
        precomputed_path=str(precomputed_path),
        shape=(128, 128, 128),
        nlod=3,
        min_chunk_size=(32, 32, 32),
        parallel=1,
    )
    print("Conversion complete.")
else:
    print("Precomputed output already exists -- skipping conversion.")

Precomputed output already exists -- skipping conversion.


## 4. Visualize in Neuroglancer


## 5. Visualize in Vitessce

`mesh_ids`, `segment_colors`, `initial_camera_state` can be adjusted for a dataset.

In [4]:
# with segments=None I one encounters this bug https://github.com/vitessce/vitessce-python/issues/517
from tissue_map_tools.view import compute_initial_camera_state
from tissue_map_tools.vitessce_configs.layer_specs import SegmentationLayerSpec
from tissue_map_tools.vitessce_configs.neuroglancer_config_builder import build_neuroglancer_config


# segments = ["612", "3351", "4328", "6531", "8446"]

initial_camera_state = compute_initial_camera_state(
    data_path=str(precomputed_path),
    # segments=segments,
)

vc = build_neuroglancer_config(
    name="Precomputed data",
    segmentations=[
        SegmentationLayerSpec(
            file_uid="segmentation", 
            local_path=str(precomputed_path), 
            # segments=segments
        ),
    ],
  
    initial_camera_state=initial_camera_state,
    # use_web_app = True
)

vc

100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 11286/11286 [00:01<00:00, 6958.43it/s]
